In [1]:
import nest_asyncio
nest_asyncio.apply()
# 중첩 허용: 이미 돌고 있는 루프안에서 또 루프를 돌려도 괜찮게 규칙을 완화

## 데이터 로드

### SimpleWebPageReader
- 웹페이지를 쉽게 가져오도록 도와주는 클래스

In [2]:
from llama_index.readers.web import SimpleWebPageReader

# 웹페이지 로드 (불러오기)
documents = SimpleWebPageReader().load_data(urls=['https://www.naver.com'])

In [3]:
documents

[Document(id_='b7220676-8fff-4327-a2c0-76faf043fdd5', embedding=None, metadata={'url': 'https://www.naver.com'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='   <!doctype html> <html lang="ko" class="fzoom"> <head> <meta charset="utf-8"> <meta name="Referrer" content="origin"> <meta http-equiv="X-UA-Compatible" content="IE=edge"> <meta name="viewport" content="width=1190"> <title>NAVER</title> <meta name="apple-mobile-web-app-title" content="NAVER"/> <meta name="robots" content="index,nofollow"/> <meta name="description" content="네이버 메인에서 다양한 정보와 유용한 컨텐츠를 만나 보세요"/> <meta property="og:title" content="네이버"> <meta property="og:url" content="https://www.naver.com/"> <meta property="og:image" content="https://s.pstatic.net/static/www/mobile/edit/2016/0705/mobile_212852414260.png"> <meta property="og:description" content="네이버 메인에서 다양한

## 유튜브 자막 가져오기

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
from llama_index.core import Document

def get_youtube_document(video_url):
    """유튜브 url을 받아 자막을 가져온 뒤 라마인덱스 document 객체로 반환"""
    try:
        video_id = '' # 1. url에서 비디오id를 추출
        if 'v=' in video_url:
            video_id = video_url.split('v=')[1].split('&')[0]
# 공유 눌렀을 때 -> https://www.youtube.com/watch?v=y46mauj6Lxw
        elif 'youtu.be/' in video_url:
            video_id = video_url.split('youtu.be/')[1].split('?')[0]

        if not video_id:
            print('비디오 id를 찾을 수 없습니다!')
            return []
        
        # 2. 자막 가져오기
        yt = YouTubeTranscriptApi()
        transcript_list = yt.list(video_id)

        # 3. 언어 선택 (한국어 -> 영어 순, 없으면 자동생성 가능한 것)
        try:
            transcript = transcript_list.find_transcript(['ko', 'en'])
        except:
            # 첫 번째로 잡히는 자막
            transcript = next(iter(transcript_list))

        transcript_data = transcript.fetch()

        # 4. 텍스트 추출 및 결합
        full_text = ''.join([entry.text for entry in transcript_data])

        # 5. 라마인덱스의 도큐먼트 객체 생성 및 반환
        return [Document(text=full_text, metadata={'video_url':video_url, 'video_id':video_id})]
    
    except Exception as e:
        print(f'에러 발생 : {e}')
        return []
    
documents = get_youtube_document('https://youtu.be/y46mauj6Lxw?si=TZXZrFguYrWpcpBe')

print(documents)

[Document(id_='bde78eef-5637-4b0f-a20a-094f1917aebb', embedding=None, metadata={'video_url': 'https://youtu.be/y46mauj6Lxw?si=TZXZrFguYrWpcpBe', 'video_id': 'y46mauj6Lxw'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='이곡들은 왜 자곡가들끼리 싸우나요?노멀라이저 버서스 브로큰 너즈사사쿠의 UK 버서스 TJ 행네일DJ 갱키 버서스 그램 블랙키vs이크로 DJ 토토토 vs 토토제아비 vs 타슈 다 동일이잖아.이게 이런 게임을 최근에 입문하신분들은 작곡과 이름에 써 있는 거에대해서 의분을 가지시는 분들도 있을것 같아. 예를 들면은 합작할 때있잖아요. A and B 이렇게 써있을 때도 있고 A 쉼표 B 이렇게써 있을 때도 있고 A미 이렇게 써있을 때도 있고 A버스 B 이렇게 써있을 때도 있고 그지? 되게많거든요. 뭐 플라티나라랩처럼 A땡땡 B 이런 경우도 있고 A B이럴 때도 있고 요런 느낌으로 AB가 되게 많잖아요. 합장 명의가이렇게 표기법에 따라 가지고 다른게뭐냐 하나하나 있을 거 아니냐라고생각들을 우리는 많이 하고 그래서대충 규칙성을 찾아냈죠. 아, A처럼둘이 동등하게 써 있는 경우는 뭔가둘이 이렇게 동등하게 합작해 가지고일반적으로 나온 거고 A 쉼표 B도그렇고 Ait B 같은 경우는 뭐B가 조금 도와줘 가지고 A가 주인데B가 도와주 그런 건가 보다라고 대충믿는 게임 하는 사람들은 규칙성을생각해 가지고 자기 나름대로 생각을하고 있을 텐데 지난번에 플라티라에공방 갔을 때 와랑 님도 그때 아마방송에서 말씀하셨나 하셨던

In [5]:
print(documents[0].text)

이곡들은 왜 자곡가들끼리 싸우나요?노멀라이저 버서스 브로큰 너즈사사쿠의 UK 버서스 TJ 행네일DJ 갱키 버서스 그램 블랙키vs이크로 DJ 토토토 vs 토토제아비 vs 타슈 다 동일이잖아.이게 이런 게임을 최근에 입문하신분들은 작곡과 이름에 써 있는 거에대해서 의분을 가지시는 분들도 있을것 같아. 예를 들면은 합작할 때있잖아요. A and B 이렇게 써있을 때도 있고 A 쉼표 B 이렇게써 있을 때도 있고 A미 이렇게 써있을 때도 있고 A버스 B 이렇게 써있을 때도 있고 그지? 되게많거든요. 뭐 플라티나라랩처럼 A땡땡 B 이런 경우도 있고 A B이럴 때도 있고 요런 느낌으로 AB가 되게 많잖아요. 합장 명의가이렇게 표기법에 따라 가지고 다른게뭐냐 하나하나 있을 거 아니냐라고생각들을 우리는 많이 하고 그래서대충 규칙성을 찾아냈죠. 아, A처럼둘이 동등하게 써 있는 경우는 뭔가둘이 이렇게 동등하게 합작해 가지고일반적으로 나온 거고 A 쉼표 B도그렇고 Ait B 같은 경우는 뭐B가 조금 도와줘 가지고 A가 주인데B가 도와주 그런 건가 보다라고 대충믿는 게임 하는 사람들은 규칙성을생각해 가지고 자기 나름대로 생각을하고 있을 텐데 지난번에 플라티라에공방 갔을 때 와랑 님도 그때 아마방송에서 말씀하셨나 하셨던 거 같은데작곡가분들도 크게 생각 없이쓰더라고요. 이거에 대해서는 저희끼리아무리 파야 작곡가마다도 쓰는뉘앙스가 다를 거기 때문에 사실 별로의미는 없고 그냥 특수하게 썼다정도에 가깝지 않을까? 그냥 좀 있어보이니까 이런 거 아닐까라고 생각을합니다. 근데 다른 이런 표기들AMB, A신표 이런 것들을 보통보면은 되게 여러 가지 합작의 형태가있잖아요. 한 명이 멜로디를 만든다음에 한 명이 편곡을 하고 뭐이렇게 티키타카하면서 이렇게만든다든지 한 명이 어디서부터어디까지 뭐 악기 연주를 하고 한명도 어디서부터 어디까지 악기를 맡은다음에 둘이 합쳐 가지고 편곡을 같이하고 뭐 이런 형태가 있을 수있잖아. 근데 이것도 뭐 작곡가분들이아니라고 하면 아닌 거긴 한데 확실한거 지금까지 게임 

## 3. 위키피디아 읽어오기

In [6]:
from llama_index.readers.wikipedia import WikipediaReader

reader = WikipediaReader()

documents = reader.load_data(pages=['Python'])

# 여러가지 종류의 노드파서 알아보기

- 노드파서 : 라마인덱스에서 문서를 작은 단위의 노드로 변환하는 역할을 한다.

## 4. Simple Node Parser (Text Splitter)

In [7]:
from llama_index.core import Document
from llama_index.core.node_parser import TokenTextSplitter, SentenceSplitter

In [8]:
text = """
LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.

주요 구성 요소는 다음과 같습니다:
Documents는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.
Nodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과 같습니다:
먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.
마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.

LlamaIndex는 다양한 인덱스 유형을 제공합니다. VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.
또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.
"""

- TokenTextSplitter : 토큰 단위로 텍스트를 분할하는 클래스
    - LLM의 토큰 제한을 고려할 때 유용하다.

In [9]:
token_splitter = TokenTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
)

token_nodes = token_splitter.split_text(text)

In [10]:
token_nodes

['LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.\n\n주요 구성 요소는 다음과 같습니다:\nDocuments는 원시 데이터를 표현하는 기본',
 '같습니다:\nDocuments는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.\nNodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.\n\n데이터 처리 과정은 다음과',
 '처리할 수 있는 크기입니다.\n\n데이터 처리 과정은 다음과 같습니다:\n먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.\n마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.\n\nLlamaIndex는 다양한',
 '검색이 가능하도록 합니다.\n\nLlamaIndex는 다양한 인덱스 유형을 제공합니다. VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.\n또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색',
 '등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.']

In [11]:
for i, node in enumerate(token_nodes, start=1):
    print()
    print(f'Node {i} : ')
    print(node)


Node 1 : 
LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.

주요 구성 요소는 다음과 같습니다:
Documents는 원시 데이터를 표현하는 기본

Node 2 : 
같습니다:
Documents는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.
Nodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과

Node 3 : 
처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과 같습니다:
먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.
마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.

LlamaIndex는 다양한

Node 4 : 
검색이 가능하도록 합니다.

LlamaIndex는 다양한 인덱스 유형을 제공합니다. VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.
또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색

Node 5 : 
등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.


- SentenceSplitter : 문장 단위로 텍스트를 분할하여 자연스러운 문장 경계를 유지하면서 노드를 나눌 수 있다.

In [12]:
sentence_splitter = SentenceSplitter(
    chunk_size=100,
    chunk_overlap=20
)

sentence_nodes = sentence_splitter.split_text(text)

In [13]:
for i, node in enumerate(sentence_nodes, start=1):
    print()
    print(f'Node {i} : ')
    print(node)


Node 1 : 
LlamaIndex는 대규모 언어 모델(LLM)을 사용하여 개인 데이터를 처리하는 데이터 프레임워크입니다. 이 프레임워크는 데이터 수집부터 처리, 검색까지 전체 과정을 지원합니다.

Node 2 : 
주요 구성 요소는 다음과 같습니다:
Documents는 원시 데이터를 표현하는 기본 단위입니다. 텍스트 파일, PDF, 웹페이지 등 다양한 소스의 데이터를 포함할 수 있습니다.

Node 3 : 
Nodes는 Documents를 더 작은 단위로 분할한 것으로, LLM이 효과적으로 처리할 수 있는 크기입니다.

데이터 처리 과정은 다음과 같습니다:
먼저 데이터 로더를 사용하여 Documents를 생성합니다. 그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.

Node 4 : 
그 다음 Node Parser를 통해 Documents를 Nodes로 분할합니다.
마지막으로 인덱스를 구축하여 효율적인 검색이 가능하도록 합니다.

LlamaIndex는 다양한 인덱스 유형을 제공합니다.

Node 5 : 
VectorStoreIndex는 임베딩 기반 검색을, SummaryIndex는 요약 기반 검색을 지원합니다.
또한 하이브리드 검색, 재순위화 등 고급 검색 기능도 제공하여 검색 품질을 향상시킬 수 있습니다.


## 5. JSONNodeParser

In [14]:
from llama_index.core.node_parser import JSONNodeParser
from llama_index.core import SimpleDirectoryReader

json_docs = SimpleDirectoryReader(input_dir='./data', required_exts=['.json']).load_data()

parser = JSONNodeParser()
json_nodes = parser.get_nodes_from_documents(json_docs)

In [15]:
# JSONNode 개수
len(json_nodes)

2

In [16]:
# 첫 번째 노드의 텍스트 출력
json_nodes[0].text

'질문 RAG가 무엇인가요?\n답변 RAG는 Retrieval Augmented Generation의 약자로, 대규모 언어 모델의 환각 현상을 줄이고 최신 정보를 반영할 수 있는 기술입니다.\n참고자료 제목 RAG 논문\n참고자료 저자 홍길동\n참고자료 제목 RAG 구현 가이드\n참고자료 링크 https://example.com/rag'

In [17]:
# 두 번째 노드의 텍스트 출력
json_nodes[1].text

'질문 RAG의 장점은 무엇인가요?\n답변 RAG는 외부 지식을 활용하여 답변의 정확도를 높이고, 실시간으로 업데이트되는 정보를 반영할 수 있습니다.\n키워드 검색 기반 생성\n키워드 지식 증강\n키워드 최신 정보 반영\n사용사례 챗봇\n사용사례 질의응답 시스템\n사용사례 문서 요약'

In [18]:
# 첫 번째 노드의 메타데이터
json_nodes[0].metadata

{'file_path': 'c:\\Users\\jangc\\ai_class\\llm_labs\\data\\json_example.json',
 'file_name': 'json_example.json',
 'file_type': 'application/json',
 'file_size': 815,
 'creation_date': '2026-05-28',
 'last_modified_date': '2026-06-03'}

# 임베팅과 인덱스

In [19]:
# api 키 가져오기
from dotenv import load_dotenv

load_dotenv()

True

In [20]:
from llama_index.core.evaluation import SemanticSimilarityEvaluator # 의미적 유사도 평가

evaluator = SemanticSimilarityEvaluator()

sentence1 = """요가는 신체와 마음 모두에게 다양한 이점을 제공합니다. 
유연성, 근력, 균형감을 향상시키는 동시에 스트레스, 불안, 통증을 줄여줍니다. 
요가는 또한 숙면, 심장 건강, 전반적인 웰빙을 증진시킵니다.
운동이나 스트레스 해소 방법을 찾고 있다면, 요가가 좋은 선택이 될 수 있습니다."""

sentence2 = """Yoga provides various benefits for both body and mind.
It improves flexibility, strength, and balance while reducing stress, anxiety, and pain.
Yoga also enhances sleep quality, heart health, and overall well-being.
If you're looking for exercise or a way to relieve stress, yoga can be a great choice."""

result = evaluator.evaluate(
    response=sentence1,
    reference=sentence2
)

In [21]:
# 유사도 점수
result.score

0.8384070942299973

In [22]:
# 통과 여부 --> 기본 유사도 임계값은 0.8로 하는 편.
result.passing

True

## 6. VectorStoreIndex

- 라마인덱스에서 가장 많이 사용하는 인덱스 유형
- 문서를 벡터(임베딩)현태로 저장하고, 검색하는데 특화되어 있다.
- 저장된 벡터들 사이의 유사도를 코사인 유사도로 측정하여 효율적인 검색을 지원한다.
- RAG 시스템 구축에 아주 적합하다.

In [23]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
import os
from dotenv import load_dotenv

In [24]:
load_dotenv() # .env파일에서 환경변수 불러온다.

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY') # OpenAI API키 설정

- Settings 클래스 : 라마인덱스에서 프로그램 전체에서 사용하는 중요한 설정을 한 곳에서 관리하는 도구
    - 언어모델(LLM), 임베딩모델, 문서 처리 방식 등 쉽게 설정하고 변경할 수 있다.

In [25]:
# LLM 설정
Settings.llm = OpenAI(model='gpt-4o', temperature=0.5)

In [26]:
# 데이터 로드
documents = SimpleDirectoryReader('./data/pdf_sample2').load_data()

In [28]:
# 파일 이름 확인
for doc in documents:
    print(doc.metadata['file_name'])

NIPS-2017-attention-is-all-you-need-Paper.pdf


- VectorStoreIndex 클래스 : 라마인덱스에서 문서의 벡터 표현을 저장하고 관리하는 핵심 구소
    - 이 인덱스는 문서를 임베딩 현태로 변환하여 저장, 유사도 기반의 빠른 검색이 가능해진다.

In [29]:
# 인덱스 생성 및 데이터 임베딩
index = VectorStoreIndex.from_documents(documents, show_progress=True)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/373 [00:00<?, ?it/s]

- 쿼리 엔진 : 사용자의 질문을 받아 인덱스에서 관련 정보를 검색, 그 정보를 바탕으로 정확하고 관련성 높은 답변을 생성

In [31]:
# 쿼리 엔진
query_engine = index.as_query_engine()

In [33]:
# 쿼리 실행
query = "what is the title of the paper? let me know the title of the paper"
response = query_engine.query(query)

# 응답 출력
print(response)

The title of the paper is "Attention Is All You Need."


## 7. 로컬에 인덱스 저장

- 장기적인 활용을 위해 로컬 디렉토리에 저장하고 나중에 불러오는 과정(인덱스 영속성)이 필요하다.
- 새로운 임베딩을 생성할 필요가 없어서 기존 벡터 데이터를 그대로 활용할 수 있으므로 속도와 리소스 절약할 수 있다.

In [34]:
# 벡터 DB 관련 라마인덱스 라이브러리 불러오기
from llama_index.core import StorageContext, load_index_from_storage

# 저장할 디렉토리 지정
persist_dir = './saved_index'

# 인덱스 저장
index.storage_context.persist(persist_dir)

In [35]:
# 저장된 인덱스 불러오기 준비
storage_context = StorageContext.from_defaults(persist_dir=persist_dir)

# 저장된 인덱스 로드 (불러오기)
loaded_index = load_index_from_storage(storage_context)

In [36]:
# 불러온 인덱스로 쿼리 엔진 생성
loaded_query_engine = loaded_index.as_query_engine()

# 다른 쿼리(질문) 실행
query = "이 논문에서 제한하는 모델의 장점은 무엇인가요?"
response = loaded_query_engine.query(query)

In [37]:
print(response)

이 논문에서 제안하는 모델의 장점은 주로 복잡성을 줄이면서도 성능을 향상시킬 수 있다는 점입니다. 특히, 이 모델은 병렬화가 용이하고, 긴 의존 관계를 효과적으로 처리할 수 있으며, 계산 효율성을 높일 수 있는 특징을 가지고 있습니다. 이러한 장점들은 자연어 처리와 같은 다양한 분야에서의 적용 가능성을 높여줍니다.


In [38]:
# 질문
print(f'질문 : {query}')
print('-' * 80)

# 답변
print(f'답변 : {response}')

질문 : 이 논문에서 제한하는 모델의 장점은 무엇인가요?
--------------------------------------------------------------------------------
답변 : 이 논문에서 제안하는 모델의 장점은 주로 복잡성을 줄이면서도 성능을 향상시킬 수 있다는 점입니다. 특히, 이 모델은 병렬화가 용이하고, 긴 의존 관계를 효과적으로 처리할 수 있으며, 계산 효율성을 높일 수 있는 특징을 가지고 있습니다. 이러한 장점들은 자연어 처리와 같은 다양한 분야에서의 적용 가능성을 높여줍니다.
